### Robust Crawling

**Goal:** Build a "Spider" that can crawl through hundreds of pages and handle messy data without crashing.

### Pagination (The Loop)
Most websites split data across multiple pages. We need a loop that updates the URL automatically. We will use **Page 3** of the bookstore as our testing ground to see how to move forward to Page 4 and beyond.

### Method 1: The Iterative URL (The "For" Loop)
**Best for:** Sites where the page number is clearly in the URL.
**Logic:** We notice the pattern `page-3.html`, `page-4.html`. We can generate these mathematically.

In [12]:
import requests
from bs4 import BeautifulSoup
import time

In [13]:
#We can replace the number 2 with the placeholder {}
base_url = "https://books.toscrape.com/catalogue/page-{}.html"

#Let's  start scrapping from page 2
for page_num in range(2, 6):
    # inject the number into the url
    current_url = base_url.format(page_num)

    print(f"Scrapping: {current_url}")

    response = requests.get(current_url)

    time.sleep(2)

Scrapping: https://books.toscrape.com/catalogue/page-2.html
Scrapping: https://books.toscrape.com/catalogue/page-3.html
Scrapping: https://books.toscrape.com/catalogue/page-4.html
Scrapping: https://books.toscrape.com/catalogue/page-5.html


### Method 2: The "Next" Button (The "While" Loop)
**Best for:** When you don't know the last page number, or you are starting in the middle (like Page 2) and just want to keep going until the end.
**Logic:** We scrape Page 2, find the "next" button, extract the link to Page 4, and update our target.

In [14]:
# Start explicitly at Page 2
url = "http://books.toscrape.com/catalogue/page-2.html"

while True:
    print(f"Scrapping: {url}")
    response = requests.get(url)
    soup = BeautifulSoup(response.content, 'html.parser')

    #Extract the data
    books = soup.select(".product_pod")

    # Moving to the next pages
    next_button = soup.select_one("li.next a")

    if next_button:
        next_page_url = next_button["href"]

        # constrct the full url
        url = "https://books.toscrape.com/catalogue/" + next_page_url

        time.sleep(2)

    else:
        print("No more pages found")
        break

Scrapping: http://books.toscrape.com/catalogue/page-2.html
Scrapping: https://books.toscrape.com/catalogue/page-3.html
Scrapping: https://books.toscrape.com/catalogue/page-4.html
Scrapping: https://books.toscrape.com/catalogue/page-5.html
Scrapping: https://books.toscrape.com/catalogue/page-6.html
Scrapping: https://books.toscrape.com/catalogue/page-7.html
Scrapping: https://books.toscrape.com/catalogue/page-8.html
Scrapping: https://books.toscrape.com/catalogue/page-9.html
Scrapping: https://books.toscrape.com/catalogue/page-10.html
Scrapping: https://books.toscrape.com/catalogue/page-11.html
Scrapping: https://books.toscrape.com/catalogue/page-12.html
Scrapping: https://books.toscrape.com/catalogue/page-13.html
Scrapping: https://books.toscrape.com/catalogue/page-14.html
Scrapping: https://books.toscrape.com/catalogue/page-15.html
Scrapping: https://books.toscrape.com/catalogue/page-16.html
Scrapping: https://books.toscrape.com/catalogue/page-17.html
Scrapping: https://books.toscrape

## Handling Missing Data
In the real world, data is messy. On `books.toscrape.com`, some books have long titles that get cut off, and in a real scenario, a price or rating might be missing.

If you try to access `.text` on a tag that doesn't exist, Python throws an `AttributeError` and your script **crashes**.

### The Problem: The "NoneType" Crash

In [15]:
tag = soup.select_one(".price")

print(tag.text)

AttributeError: 'NoneType' object has no attribute 'text'

### The Solution: `try` / `except` Blocks
We wrap the extraction code in a `try` block. If it fails, Python jumps to the `except` block, assigns a default value ("N/A"), and keeps running.

In [16]:
# Dummy HTML simulating a broken book card on Page 3
html = """
<div class="product">
    <h2>The Book of Python</h2>
    <!-- Price is missing here! -->
</div>
"""

soup = BeautifulSoup(html, "html.parser")
card = soup.select_one(".product")

item = {}

try:
    item['title'] = card.select_one("h2").get_text(strip=True)
except AttributeError:
    item['title'] = "Unknown Title"

#Price
try:
    item['price'] = card.select_one(".price").get_text(strip=True)
except AttributeError:
    item['price'] = "N/A"
    print("Warning: The price is missing")

print(item)

{'title': 'The Book of Python', 'price': 'N/A'}


### Alternative: The "If" Check
For simple checks, you can verify if the element exists before asking for text.

In [5]:
price_tag = card.select_one(".price")

if price_tag:
    price = price_tag.get_text(strip=True)
else:
    price = "0.00"


## Network Reliability (Status Codes)
Sometimes the error isn't in your code; it's in the server (e.g., the website goes down, or you get banned).

We should always check `response.status_code` before feeding the content to BeautifulSoup.

In [17]:
response = requests.get(url)

# Raise an error immediately if the status is 4xx or 5xx
try:
    response.raise_for_status() 
    # If we get here, the request was successful (200)
    soup = BeautifulSoup(response.content, "html.parser")
except requests.exceptions.HTTPError as e:
    print(f"Skipping page due to network error: {e}")